## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 

In [2]:

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT_neighborhood_finder

from TCT import TCT



### Load Translator resources


In [3]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources(use_new_metakg_url=True)

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

    # generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'url': '/sipr'}


### Select endpoints for query


In [6]:
# This is an example of selecting a list of APIs for the neighborhood finder. The user can modify this list to include the APIs they want to use. The APIs in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. The user can also modify the list of predicates to use for finding the neighborhood. The predicates in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. 
# The user can also modify the list of categories to use for finding the neighborhood. 
# The categories in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph.
# if selected_APIlist is empty, use all APIs in APInames
selected_APIlist = ['Retriever',
                    'Clinical Trials KP - TRAPI 1.5.0',
                    'Drug Approvals KP - TRAPI 1.5.0',
                    'Genetics Data Provider for NCATS Biomedical Translator Reasoners',
                    'Microbiome KP - TRAPI 1.5.0',
                    'MolePro',
                    'COHD TRAPI',
                    'RTX KG2 - TRAPI 1.5.0',
                    'Text Mined Cooccurrence API',
                    'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0',
                    'CATRAX BigGIM GeneExpression Performance Phase KP - TRAPI 1.5.0',
                    ]

# add Automat API to the selected API list if it is not already in the list
for api in APInames:
    if 'Automat' in api and api not in selected_APIlist:
        selected_APIlist.append(api)
        
# selected_APIlist = []
# select a list of APIs to use and a list of predicates to use
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)
print(selected_metaKG.shape)

All_predicates = list(set(selected_metaKG['Predicate']))
All_categories = list((set(list(set(selected_metaKG['Subject']))+list(set(selected_metaKG['Object'])))))
API_withMetaKG = list(set(selected_metaKG['API']))
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(selected_metaKG[selected_metaKG['API'] == api]['Predicate']))

(20830, 5)


## Find the neighborhood of an entity from a subset of APIs 


In [ ]:
#name_resolver.lookup('BACE1', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')
#name_resolver.lookup('TP53', return_top_response=False, biolink_type='biolink:Gene',  limit=100, only_taxa='NCBITaxon:9606') # sometimes the identifiers are not in the top 1, users need to check the other returned results
#name_resolver.lookup('alzheimer disease')
name_resolver.lookup('acute myeloid leukemia', return_top_response=True, biolink_type='biolink:Disease',  limit=10) # sometimes the identifiers are not in the top 1, users need to check the other returned results


TranslatorNode(curie='MONDO:0018874', label='acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[])

In order to use the neighborhood finder, we have to look up a CURIE ID for a given term.

In [ ]:

#input_identifiers = 'MONDO:0004975'
input_identifiers = 'MONDO:0018874'

input_node_info = node_normalizer.get_normalized_nodes(input_identifiers)
input_node_info


TranslatorNode(curie='MONDO:0018874', label='acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing'], synonyms=None, curie_synonyms=None, attributes=None, taxa=None)


Neighborhood finder identifies all nodes *b* that are connected to the given node *a*, where *b* is part of a defined list of categories - returning the neighborhood of node *a*.

In [ ]:
# to exclude BioThings Explorer (BTE) TRAPI: 
input_node_id, result, result_parsed, result_ranked_by_primary_infores = TCT_neighborhood_finder.neighborhood_finder(input_identifiers,
                                                                                            node2_categories = ['biolink:Drug','biolink:SmallMolecule', 'biolink:ChemicalSubstance'],
                                                                                            APInames = select_APIs,
                                                                                            metaKG = selected_metaKG,
                                                                                            API_predicates = API_predicates)     

MONDO:0018874
Drug Approvals KP - TRAPI 1.5.0: Success!
Automat-hetionet(Trapi v1.5.0): Success!
COHD TRAPI: Success!
RTX KG2 - TRAPI 1.5.0: Success!
Automat-ctd(Trapi v1.5.0): Success!
Automat-drug-central(Trapi v1.5.0): Success!
Automat-robokop(Trapi v1.5.0): Success!
Retriever: Success!
MolePro: Success!
Clinical Trials KP - TRAPI 1.5.0: Success!
NodeNorm does not know about these identifiers: RXCUI:1791496,RXCUI:1736582,DRUGBANK:DB15060,REACT:R-ALL-9692345,CHEBI:232328,CHEBI:232616,CHEBI:233310,CHEBI:233362,CHEBI:232584,CHEBI:233318,CHEBI:233359,CHEBI:233593,GTOPDB:13607,UMLS:C5888788,UMLS:C5907931,UMLS:C5907992,UMLS:C5908001,UMLS:C5979854,CHEBI:458192,CHEBI:64208,CHEBI:391051,PUBCHEM.COMPOUND:155886736,CHEBI:37616,CHEBI:1093323,CHEBI:2360


In [ ]:
TCT_path_finder_result = TCT_neighborhood_finder.parse_results_for_neighborhood_finder(input_identifiers, result,
        start_node_categories='biolink:Disease', end_node_categories=None,
        get_node_info=True,
        scoring_method='infores')

In [ ]:
# write a result to a json file
import json
with open('TCT_neighborhood_finder_result_'+input_identifiers.replace(':', '_')+'new.json', 'w') as f:
    json.dump(TCT_path_finder_result, f)

In [ ]:
# End of the example
